In [1]:
import geopandas as gpd
import pandas as pd
import ee
import os
import numpy as np
from pathlib import Path
from shapely.geometry import mapping
import geemap

In [2]:
try:
    ee.Initialize(project='ee-biswajit001')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-biswajit001')

In [3]:
from eedata import GEETimeSeriesExtractor
from processors import TimeSeriesProcessor
from visualizer import TimeSeriesPlotter

In [4]:
cwd = Path.cwd()
farm_file_path = Path.joinpath(cwd, r'example_data\input\farm1.gpkg')
gdf = gpd.read_file(farm_file_path)

In [5]:
gdf.head()

,Shape_Area,AREA,geometry
0,0.0,16.117602,"POLYGON ((35.38742 -11.44702, 35.38624 -11.445..."


In [6]:
my_farm_geometry = geemap.gdf_to_ee(gdf).geometry()

In [7]:
# 1. Initialize the Extractor
extractor = GEETimeSeriesExtractor()

# 2. Fetch data: We want standard NDVI, but also a custom Soil Moisture proxy
raw_df = extractor.extract(
    aoi=my_farm_geometry,
    start_date='2025-06-01',
    end_date='2026-04-01',
    indices=['NDVI', 'NDTI'], 
    custom_formulas={'CUSTOM_RATIO': 'b("B3") / b("B12")'}, # Dynamic!
    apply_mask=True # Toggle cloud masking
)

# 3. Process the data: Let's aggregate by Fortnight (2 Weeks)
processor = TimeSeriesProcessor(raw_df)
processed_df = processor.aggregate(frequency='2W').df

# (Optional: You could call processor.smooth_series('NDVI') here to get your arrays for peaks/dips)

# 4. Plot the data: Let's use Plotly for an interactive dashboard
plotter = TimeSeriesPlotter(processed_df)
plotter.plot_interactive(columns_to_plot=['NDVI', 'NDTI', 'CUSTOM_RATIO'], title="Farm 402 - Fortnightly Dynamics")